# 09 · Thesis and publication figures

Simple figures made for print. Every figure:
* is sized for a journal page: **one column = 90 mm**, **two columns = 190 mm** wide;
* uses 7-8 pt text at that printed size, panel labels **(a), (b)**, units on every axis and no title (the caption says what it shows);
* gives each feature set the same colour everywhere: **grey = XYZ**, **navy = XYZ + VA-HDW**; the final model is marked *;
* is saved as **PDF** (vector) and **PNG at 600 dpi** (for Word).

Figures that show borehole locations are saved in `private/`. Nothing is refitted: every figure reads the outputs of notebooks 02-08.

| Fig. | Content |
|---|---|
| 1 | blockCV: (a) folds, (b) purge example *(private)* |
| 2 | Model comparison: Macro-F1, XYZ vs XYZ + VA-HDW |
| 3 | VA-HDW gain per model, fold by fold |
| 4 | Macro-F1 vs distance to the nearest training borehole (final model's model, both feature sets) |
| 5 | Confusion matrix and mean spatial ROC curve (final model and comparison model) |
| 6 | Held-out permutation importance of the input groups |
| 7 | New external boreholes: final vs comparison model, (a) scores, (b) per borehole vs distance |
| 8 | 3D map: P(Clay) at four depths *(private, if notebook 08 was run)* |
| S1 | Block-size sensitivity and purge (XGB, both feature sets) |

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import geopandas as gpd
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

OUT, PRIV = H.out_dir(S, "09_figures"), H.out_dir(S, "09_figures", private=True)
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
summ = pd.read_csv(H.out_dir(S, "04_results") / "scores_mean_sd.csv")
fold_tab = pd.read_csv(H.out_dir(S, "04_results") / "scores_by_fold.csv")
fin = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())
FINAL = (fin["model"], fin["features"])
CMP = (fin["comparison"]["model"], fin["comparison"]["features"]) if fin.get("comparison") else None
main = preds[(preds.multiple == 1) & (preds.purge_x == 0)]
MODELS = [m for m in list(S["cv"]["models"]) + list(S["cv"]["vahdw_only_models"]) if m in set(main.model)]
B = f"{block_m:.0f}"

# ---- style -------------------------------------------------------------------------------
MM = 1 / 25.4
W1, W2 = 90 * MM, 190 * MM
FCOL = {"XYZ": "#9E9E9E", "XYZ_VAHDW": "#17365C"}
FNICE = {"XYZ": "XYZ", "XYZ_VAHDW": "XYZ + VA-HDW"}
NICE = {"RF": "RF", "XGB": "XGBoost", "MLP": "MLP", "GWRF": "GWRF", "GWXGB": "GWXGBoost", "GBARNN": "GBARNN"}
MAGENTA, NAVY = "#A0004E", "#17365C"
FOLD_COL = ["#2C6FBB", "#F28E2B", "#59A14F", "#C0000B", "#7B5CA8"]
plt.rcParams.update({"font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "pdf.fonttype": 42, "savefig.dpi": 600, "savefig.bbox": "tight", "figure.dpi": 150})

def label(ax, s, x=-0.14, y=1.04):
    ax.text(x, y, f"({s})", transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")

def save(fig, name, private=False):
    folder = PRIV if private else OUT
    for ext in ("pdf", "png"):
        fig.savefig(folder / f"{name}.{ext}")
    print("saved", folder / f"{name}.pdf")

name_of = lambda r: f"{NICE[r[0]]} ({FNICE[r[1]]})"
km = lambda v, _: f"{v / 1000:.0f}"
print("Final model:", name_of(FINAL), "| comparison:", name_of(CMP) if CMP else None)

## Fig. 1 · blockCV folds and purge example (private)

In [ ]:
blk = gpd.read_file(H.out_dir(S, "02_blockcv", private=True) / "blocks_x1.gpkg")
bh = cells.groupby("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].first()
bh["fold"] = bh.index.map(folds).astype(int)
pts = gpd.GeoSeries(gpd.points_from_xy(bh.UTM_Easting_m, bh.UTM_Northing_m), crs=blk.crs)
occupied = blk[blk.intersects(pts.union_all())]
FOLD = 1
fig, axes = plt.subplots(1, 2, figsize=(W2, W2 * 0.45))
for ax in axes:
    for k in sorted(bh.fold.unique()):
        occupied[occupied.folds == k].plot(ax=ax, color=FOLD_COL[k - 1], alpha=0.2, edgecolor=FOLD_COL[k - 1], linewidth=0.4)
        b = bh[bh.fold == k]
        ax.scatter(b.UTM_Easting_m, b.UTM_Northing_m, s=7, color=FOLD_COL[k - 1], linewidth=0, label=f"Fold {k}", zorder=3)
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km); ax.set_aspect("equal")
    ax.set_xlabel("Easting (km, UTM 48N)"); ax.set_ylabel("Northing (km)")
test, train = bh[bh.fold == FOLD], bh[bh.fold != FOLD]
d = H.nearest_distance(train[["UTM_Easting_m", "UTM_Northing_m"]].values, test[["UTM_Easting_m", "UTM_Northing_m"]].values)
removed = train[d < block_m]
for x, y in test[["UTM_Easting_m", "UTM_Northing_m"]].values:
    axes[1].add_patch(plt.Circle((x, y), block_m, fill=False, color="#777777", linewidth=0.5, zorder=2))
axes[1].scatter(removed.UTM_Easting_m, removed.UTM_Northing_m, s=12, marker="x", color="black", linewidth=0.7, zorder=4)
h, l = axes[0].get_legend_handles_labels()
axes[0].legend(h, l, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=5, handletextpad=0.2, columnspacing=0.8)
axes[1].legend([Line2D([], [], color="#777777", lw=0.5), Line2D([], [], marker="x", ls="", color="black")],
               [f"{B} m purge circle (fold {FOLD})", f"removed training borehole ({len(removed)})"],
               loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
label(axes[0], "a", y=1.1); label(axes[1], "b", y=1.1)
fig.tight_layout()
save(fig, "fig01_blockcv_folds", private=True)

## Fig. 2 · Model comparison

In [ ]:
s = summ[(summ.multiple == 1) & (summ.purge_x == 0) & (summ.cells == "all")]
fig, ax = plt.subplots(figsize=(W2 * 0.75, W1 * 0.6))
x = np.arange(len(MODELS))
for j, f in enumerate(("XYZ", "XYZ_VAHDW")):
    t = s[s.features == f].set_index("model").reindex(MODELS)
    xs = x + (j - 0.5) * 0.38
    ax.bar(xs, t.macro_f1_mean, width=0.36, yerr=t.macro_f1_sd, color=FCOL[f], label=FNICE[f], capsize=2,
           error_kw={"elinewidth": 0.6, "capthick": 0.6})
    for xi, v, m in zip(xs, t.macro_f1_mean, MODELS):
        if np.isfinite(v):
            ax.text(xi, 0.02, f"{v:.3f}" + (" *" if (m, f) == FINAL else ""), ha="center", va="bottom",
                    rotation=90, fontsize=6, color="white")
ax.set_xticks(x, [NICE[m] for m in MODELS])
ax.set_ylabel("Macro-F1 (mean ± SD, 5 outer folds)"); ax.set_ylim(0, 1)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
save(fig, "fig02_model_comparison")
display(s[["model", "features", "macro_f1_mean", "macro_f1_sd", "accuracy_mean", "balanced_accuracy_mean",
           "roc_auc_mean", "brier_mean"]].round(3))

## Fig. 3 · VA-HDW gain, fold by fold
Each dot = one outer fold: Macro-F1 with XYZ + VA-HDW minus Macro-F1 with XYZ, on the same test boreholes. Bar = mean ± 1 SE. Rule (notebook 04): VA-HDW helps only if the mean is above one SE.

In [ ]:
f = fold_tab[(fold_tab.cells == "all") & (fold_tab.multiple == 1) & (fold_tab.purge_x == 0)]
t = f.pivot_table(index=["model", "outer_fold"], columns="features", values="macro_f1").dropna()
gain = H.paired_gain(f)
pm = [m for m in MODELS if m in set(gain.model)]
fig, ax = plt.subplots(figsize=(W1, W1 * 0.7))
for i, m in enumerate(pm):
    dd = (t.loc[m, "XYZ_VAHDW"] - t.loc[m, "XYZ"]).values
    g = gain.set_index("model").loc[m]
    ax.errorbar(i, g.mean_gain, yerr=g.se_gain, fmt="s", color=NAVY, markersize=4, capsize=3, elinewidth=0.8, zorder=3)
    ax.scatter(np.full(len(dd), i) + np.linspace(-0.12, 0.12, len(dd)), dd, s=6, color="#777777", zorder=2, linewidth=0)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xticks(range(len(pm)), [NICE[m] for m in pm])
ax.set_ylabel("Macro-F1 gain from VA-HDW")
save(fig, "fig03_vahdw_gain")
display(gain.round(3))

## Fig. 4 · By distance to the nearest training borehole

In [ ]:
bands = pd.read_csv(H.out_dir(S, "04_results") / "scores_by_distance.csv")
bands = bands[(bands.cells == "all") & (bands.model == FINAL[0])]
order = list(dict.fromkeys(bands.band))
fig, ax = plt.subplots(figsize=(W1, W1 * 0.7))
for f_, g in bands.groupby("features"):
    g = g.set_index("band").reindex(order)
    ax.plot(range(len(order)), g.macro_f1, "-o", color=FCOL[f_], markersize=3, label=f"{NICE[FINAL[0]]} ({FNICE[f_]})")
ax.set_xticks(range(len(order)), order)
n = bands.groupby("band").n.max().reindex(order)
for i, v in enumerate(n):
    ax.text(i, 0.02, f"n = {v}", ha="center", fontsize=6, color="#555555")
ax.set_ylim(0, 1); ax.set_ylabel("Macro-F1 (pooled test cells)"); ax.set_xlabel("Distance to the nearest training borehole")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0))
save(fig, "fig04_distance_bands")

## Fig. 5 · Confusion matrix and mean spatial ROC curve

In [ ]:
for run, tag in ((FINAL, "final"), (CMP, "comparison")):
    if run is None or not ((main.model == run[0]) & (main.features == run[1])).any():
        continue
    d = main[(main.model == run[0]) & (main.features == run[1])]
    cm = confusion_matrix(d.y, d.y_pred, labels=[0, 1])
    fig, axes = plt.subplots(1, 2, figsize=(W2 * 0.75, W1 * 0.62), gridspec_kw={"width_ratios": [1, 1.1]})
    ax = axes[0]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({cm[i, j] / cm[i].sum():.0%})", ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max() / 2 else NAVY)
    ax.set_xticks([0, 1], ["Sand", "Clay"]); ax.set_yticks([0, 1], ["Sand", "Clay"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Observed")
    for s_ in ax.spines.values():
        s_.set_visible(False)
    ax = axes[1]
    grid = np.linspace(0, 1, 201); tprs, aucs = [], []
    for k, g in d.groupby("outer_fold"):
        if g.y.nunique() == 2:
            fpr, tpr, _ = roc_curve(g.y, g.p_clay)
            ax.plot(fpr, tpr, color="#BBBBBB", linewidth=0.5)
            tprs.append(np.interp(grid, fpr, tpr)); tprs[-1][0] = 0.0
            aucs.append(roc_auc_score(g.y, g.p_clay))
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    ax.plot(grid, mean_tpr, color=FCOL[run[1]] if run[1] == "XYZ_VAHDW" else "#555555", linewidth=1.4)
    sd_auc = np.std(aucs, ddof=1) if len(aucs) > 1 else 0.0
    ax.text(0.97, 0.04, f"Mean AUC = {np.mean(aucs):.3f}\n(SD {sd_auc:.3f}, {len(aucs)} folds)", ha="right", va="bottom",
            transform=ax.transAxes, fontsize=7)
    ax.plot([0, 1], [0, 1], linestyle="--", color="#999999", linewidth=0.6)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (Clay)")
    label(axes[0], "a", x=-0.3); label(axes[1], "b", x=-0.25)
    fig.tight_layout()
    save(fig, f"fig05_confusion_roc_{tag}_{run[0]}_{run[1]}")

## Fig. 6 · Held-out permutation importance

In [ ]:
perm = pd.read_csv(H.out_dir(S, "06_xai") / "permutation_importance.csv")
perm = perm[(perm.cells == "all") & (perm.kind == "group") | (perm.cells == "all") & (perm.input == "Elevation")]
INPUTS = ["Location (E+N)", "Elevation", "VA-HDW d", "VA-HDW Φ", "VA-HDW S", "VA-HDW all"]
runs = [r for r in (FINAL, CMP) if r is not None and ((perm.model == r[0]) & (perm.features == r[1])).any()]
fig, ax = plt.subplots(figsize=(W1, W1 * 0.75))
for j, r in enumerate(runs):
    g = perm[(perm.model == r[0]) & (perm.features == r[1])].set_index("input").reindex(INPUTS)
    y = np.arange(len(INPUTS)) + (j - (len(runs) - 1) / 2) * 0.38
    ax.barh(y, g["mean"], height=0.36, xerr=g["std"], color=FCOL[r[1]], label=name_of(r) + (" *" if r == FINAL else ""),
            error_kw={"elinewidth": 0.6})
ax.set_yticks(range(len(INPUTS)), INPUTS); ax.invert_yaxis(); ax.axvline(0, color="black", linewidth=0.6)
ax.set_xlabel("Drop in balanced accuracy (mean ± SD, folds)")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0))
save(fig, "fig06_permutation_importance")

## Fig. 7 · New external boreholes (notebook 07)

In [ ]:
ext_scores = H.out_dir(S, "07_new_external") / "new_external_scores.csv"
ext_bh = H.out_dir(S, "07_new_external", private=True) / "new_external_by_borehole.csv"
if ext_scores.exists():
    res = pd.read_csv(ext_scores)
    a = res[(res.group == "all") & (res.cells == "all")].set_index("model")
    per = pd.read_csv(ext_bh)
    names = list(a.index)
    fig, axes = plt.subplots(1, 2, figsize=(W2, W1 * 0.62))
    ax = axes[0]
    METRICS = ["macro_f1", "balanced_accuracy", "roc_auc", "brier"]
    x = np.arange(len(METRICS))
    for j, nm in enumerate(names):
        feat = "XYZ_VAHDW" if nm.endswith("XYZ_VAHDW") else "XYZ"
        xs = x + (j - (len(names) - 1) / 2) * 0.38
        v = a.loc[nm, METRICS].values.astype(float)
        ax.bar(xs, v, width=0.36, color=FCOL[feat], label=f"{NICE[nm.split(' | ')[0]]} ({FNICE[feat]})")
        for xi, vi in zip(xs, v):
            ax.text(xi, vi + 0.01, f"{vi:.2f}", ha="center", fontsize=6)
    ax.set_xticks(x, ["Macro-F1", "Bal. acc.", "ROC-AUC", "Brier ↓"]); ax.set_ylim(0, 1); ax.set_ylabel("Score")
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0))
    ax = axes[1]
    for nm in names:
        feat = "XYZ_VAHDW" if nm.endswith("XYZ_VAHDW") else "XYZ"
        g = per[per.model == nm]
        ax.scatter(g.dist_nearest_train_m, g.accuracy, s=10, color=FCOL[feat], linewidth=0,
                   label=f"{NICE[nm.split(' | ')[0]]} ({FNICE[feat]})")
    ax.set_xlabel("Distance to the nearest training borehole (m)"); ax.set_ylabel("Accuracy per borehole"); ax.set_ylim(0, 1.02)
    label(axes[0], "a"); label(axes[1], "b")
    fig.tight_layout()
    save(fig, "fig07_new_external")
    display(a[["n", "accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "brier"]].round(3))
else:
    print("Notebook 07 has not been run yet.")

## Fig. 8 · 3D map at four depths (private)

In [ ]:
maps = sorted(H.out_dir(S, "08_map", private=True).glob("map_3d_points_*.csv"))
if maps:
    grid = pd.read_csv(maps[-1], usecols=["X", "Y", "depth_top_m", "p_clay"])
    fig, axes = plt.subplots(1, 4, figsize=(W2, W2 * 0.3), sharey=True)
    for ax, depth in zip(axes, [1.0, 5.0, 10.0, 14.0]):
        g = grid[np.isclose(grid.depth_top_m, depth)]
        sc = ax.scatter(g.X, g.Y, c=g.p_clay, s=0.3, cmap="RdYlBu_r", vmin=0, vmax=1, linewidth=0, rasterized=True)
        ax.set_aspect("equal"); ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
        ax.set_xlabel("Easting (km)"); ax.text(0.02, 0.98, f"{depth:g} m", transform=ax.transAxes, va="top", fontsize=7)
    axes[0].set_ylabel("Northing (km)")
    fig.colorbar(sc, ax=axes, fraction=0.02, label="P(Clay)")
    save(fig, "fig08_map_depths", private=True)
else:
    print("Notebook 08 has not been run yet.")

## Fig. S1 · Block-size sensitivity and purge

In [ ]:
ss = summ[(summ.cells == "all") & summ.model.isin(S["cv"]["sensitivity_models"])]
fig, axes = plt.subplots(1, 2, figsize=(W2 * 0.75, W1 * 0.6), sharey=True)
for f_, g in ss[ss.purge_x == 0].groupby("features"):
    g = g.sort_values("block_m")
    axes[0].errorbar(g.block_m, g.macro_f1_mean, yerr=g.macro_f1_sd, fmt="-o", color=FCOL[f_], markersize=3, capsize=2,
                     elinewidth=0.6, label=FNICE[f_])
pp = summ[(summ.cells == "all") & (summ.multiple == 1) & summ.model.isin(S["cv"]["purge_models"])]
for f_, g in pp.groupby("features"):
    g = g.sort_values("purge_m")
    axes[1].errorbar(g.purge_m, g.macro_f1_mean, yerr=g.macro_f1_sd, fmt="-o", color=FCOL[f_], markersize=3, capsize=2,
                     elinewidth=0.6, label=FNICE[f_])
axes[0].set_xlabel("Block size (m)"); axes[1].set_xlabel("Purge distance (m)")
axes[0].set_ylabel(f"Macro-F1, {NICE[S['cv']['sensitivity_models'][0]]} (mean ± SD)"); axes[0].set_ylim(0, 1)
axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
label(axes[0], "a"); label(axes[1], "b")
fig.tight_layout()
save(fig, "figS1_sensitivity_purge")
print(f"Finished in {time.time() - T0:.0f} s.")